In [1]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
train = pd.read_csv(ROOT / "data/processed/train.csv")
feat_cols = [f"f{i}" for i in range(63)]
X = train[feat_cols].to_numpy(dtype=np.float64)
y = train["label"].astype(str).to_numpy()

def mirror(X):
    Xm = X.copy()
    Xm[:, 0::3] *= -1
    return Xm

TIPS = [4, 8, 12, 16, 20]                         # fingertip points
CHAINS = [(0,1,2,3,4), (0,5,6,7,8), (0,9,10,11,12),
          (0,13,14,15,16), (0,17,18,19,20)]       # points of each finger

def add_engineered_features(X):
    P = X.reshape(len(X), 21, 3)
    feats = []
    for t in TIPS:                                 # 5: distance from tip to wrist
        feats.append(np.linalg.norm(P[:, t], axis=1))
    for i in range(5):                             # 10: distance from tip to tip
        for j in range(i + 1, 5):
            feats.append(np.linalg.norm(P[:, TIPS[i]] - P[:, TIPS[j]], axis=1))
    for chain in CHAINS:                           # 15: joint angles
        for k in range(1, 4):
            a = P[:, chain[k-1]] - P[:, chain[k]]
            b = P[:, chain[k+1]] - P[:, chain[k]]
            cos = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-9)
            feats.append(np.arccos(np.clip(cos, -1, 1)))
    return np.hstack([X, np.column_stack(feats)])

X93 = add_engineered_features(X)
print("shape:", X.shape, "->", X93.shape)
print("new features unchanged by mirror?",
      np.allclose(add_engineered_features(mirror(X))[:, 63:], X93[:, 63:]))
print("any NaN in a feature?", np.isnan(X93).any())

shape: (2165, 63) -> (2165, 93)
new features unchanged by mirror? True
any NaN in a feature? False


In [2]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_model(gamma):
    return Pipeline([("scaler", StandardScaler()),
                     ("svm", SVC(kernel="rbf", C=100, gamma=gamma))])

variants = {
    "1 raw63 + mirror (v2)  g=0.01": (False, 0.01),
    "2 eng93 + mirror  g=0.01":      (True,  0.01),
    "3 eng93 + mirror  g=0.0068":    (True,  0.0068),
}

rows = []
for fold, (tr, va) in enumerate(skf.split(X, y), start=1):
    Xtr, ytr, Xva, yva = X[tr], y[tr], X[va], y[va]
    Xtr_aug = np.vstack([Xtr, mirror(Xtr)])
    ytr_aug = np.concatenate([ytr, ytr])
    for name, (use_eng, gamma) in variants.items():
        f = add_engineered_features if use_eng else (lambda A: A)
        m = make_model(gamma).fit(f(Xtr_aug), ytr_aug)
        for side, Xv in [("right", Xva), ("left", mirror(Xva))]:
            p = m.predict(f(Xv))
            rows.append({"fold": fold, "variant": name, "hand": side,
                         "acc": accuracy_score(yva, p),
                         "macro_f1": f1_score(yva, p, average="macro")})

res2 = pd.DataFrame(rows)
print(res2.groupby(["variant", "hand"])[["acc", "macro_f1"]].mean().round(4))

(ROOT / "results").mkdir(exist_ok=True)
res2.to_csv(ROOT / "results/advanced_features_cv.csv", index=False)

piv = res2[res2.hand == "right"].pivot(index="fold", columns="variant", values="acc")
print(piv.round(4))

                                        acc  macro_f1
variant                       hand                   
1 raw63 + mirror (v2)  g=0.01 left   0.9312    0.9274
                              right  0.9316    0.9279
2 eng93 + mirror  g=0.01      left   0.9483    0.9458
                              right  0.9483    0.9458
3 eng93 + mirror  g=0.0068    left   0.9460    0.9429
                              right  0.9460    0.9429
variant  1 raw63 + mirror (v2)  g=0.01  2 eng93 + mirror  g=0.01  \
fold                                                               
1                               0.9353                    0.9492   
2                               0.9261                    0.9515   
3                               0.9215                    0.9330   
4                               0.9353                    0.9515   
5                               0.9400                    0.9561   

variant  3 eng93 + mirror  g=0.0068  
fold                                 
1              

In [3]:
import sys, joblib
sys.path.append(str(ROOT))
from sklearn.preprocessing import FunctionTransformer
from src.features import add_engineered_features as eng_src

print("src function == notebook function?", np.allclose(eng_src(X), add_engineered_features(X)))

test = pd.read_csv(ROOT / "data/processed/test.csv")
Xte = test[feat_cols].to_numpy(dtype=np.float64)
yte = test["label"].astype(str).to_numpy()

def make_final(gamma):
    return Pipeline([("eng", FunctionTransformer(eng_src)),
                     ("scaler", StandardScaler()),
                     ("svm", SVC(kernel="rbf", C=100, gamma=gamma,
                                 probability=True, random_state=42))])

X_aug = np.vstack([X, mirror(X)])
y_aug = np.concatenate([y, y])
final_v3  = make_final(0.01).fit(X_aug, y_aug)      # chosen by CV
final_v3b = make_final(0.0068).fit(X_aug, y_aug)    # also built, so it will be reported

core = joblib.load(ROOT / "models/best_model_core_v1.joblib")
v2   = joblib.load(ROOT / "models/best_model_v2_mirror.joblib")

models = {
    "core v1 (63, no mirror)":          core,
    "v2 (63 + mirror)":                 v2,
    "v3 (93 + mirror, g=0.01) CHOSEN":  final_v3,
    "v3b (93 + mirror, g=0.0068)":      final_v3b,
}
out = []
for name, m in models.items():
    for side, Xt in [("right", Xte), ("left", mirror(Xte))]:
        p = m.predict(Xt)
        out.append({"model": name, "hand": side,
                    "test_acc": accuracy_score(yte, p),
                    "test_macro_f1": f1_score(yte, p, average="macro")})
out = pd.DataFrame(out).round(4)
print(out.to_string(index=False))

out.to_csv(ROOT / "results/advanced_features_test.csv", index=False)
joblib.dump(final_v3, ROOT / "models/best_model_v3_eng.joblib")

src function == notebook function? True


d:\CS 5 Course\ML(Lab)\Project\psl-translator\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
d:\CS 5 Course\ML(Lab)\Project\psl-translator\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


                          model  hand  test_acc  test_macro_f1
        core v1 (63, no mirror) right    0.9502         0.9482
        core v1 (63, no mirror)  left    0.7972         0.7907
               v2 (63 + mirror) right    0.9377         0.9355
               v2 (63 + mirror)  left    0.9377         0.9355
v3 (93 + mirror, g=0.01) CHOSEN right    0.9324         0.9277
v3 (93 + mirror, g=0.01) CHOSEN  left    0.9324         0.9277
    v3b (93 + mirror, g=0.0068) right    0.9342         0.9294
    v3b (93 + mirror, g=0.0068)  left    0.9342         0.9294


['d:\\CS 5 Course\\ML(Lab)\\Project\\psl-translator\\models\\best_model_v3_eng.joblib']

In [4]:
from sklearn.model_selection import StratifiedGroupKFold

groups = train["signer_id"].astype(str).to_numpy()
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

variants = {
    "A raw63, no mirror (core style)": (False, False),
    "B raw63 + mirror (v2)":           (False, True),
    "C eng93 + mirror (v3)":           (True,  True),
    "D eng93, no mirror":              (True,  False),
}

rows = []
for fold, (tr, va) in enumerate(sgkf.split(X, y, groups), start=1):
    assert set(groups[tr]).isdisjoint(groups[va]), "signer overlap!"
    Xtr, ytr, Xva, yva = X[tr], y[tr], X[va], y[va]
    for name, (use_eng, use_mirror) in variants.items():
        if use_mirror:
            Xa, ya = np.vstack([Xtr, mirror(Xtr)]), np.concatenate([ytr, ytr])
        else:
            Xa, ya = Xtr, ytr
        f = add_engineered_features if use_eng else (lambda A: A)
        m = make_model(0.01).fit(f(Xa), ya)
        for side, Xv in [("right", Xva), ("left", mirror(Xva))]:
            p = m.predict(f(Xv))
            rows.append({"fold": fold, "variant": name, "hand": side,
                         "acc": accuracy_score(yva, p),
                         "macro_f1": f1_score(yva, p, average="macro")})

res3 = pd.DataFrame(rows)
print(res3.groupby(["variant", "hand"])[["acc", "macro_f1"]].mean().round(4))
res3.to_csv(ROOT / "results/advanced_grouped_cv.csv", index=False)

print(res3[res3.hand == "right"].pivot(index="fold", columns="variant", values="acc").round(4))

                                          acc  macro_f1
variant                         hand                   
A raw63, no mirror (core style) left   0.7725    0.7622
                                right  0.9252    0.9222
B raw63 + mirror (v2)           left   0.9311    0.9268
                                right  0.9311    0.9268
C eng93 + mirror (v3)           left   0.9499    0.9479
                                right  0.9499    0.9479
D eng93, no mirror              left   0.8871    0.8796
                                right  0.9458    0.9435
variant  A raw63, no mirror (core style)  B raw63 + mirror (v2)  \
fold                                                              
1                                 0.9465                 0.9279   
2                                 0.9229                 0.9369   
3                                 0.9137                 0.9233   
4                                 0.9213                 0.9371   
5                                 0.92